# Cross-species embedding — scPRINT `ujzjjsi3` with pig genes

Objective: run checkpoint `ujzjjsi3.ckpt` on the shared cat/tiger benchmark
after remapping the existing mouse-gene matrix to high-confidence one-to-one
pig orthologs, then compute the same scIB scores as the mouse run.

The mapping was downloaded manually from Ensembl BioMart before submission.
This notebook performs no download and fails if the local mapping is absent.
All cells use the pig organism token (`NCBITaxon:9823`).


In [1]:
import os
from pathlib import Path

os.environ.update({
    "HF_HUB_OFFLINE": "1",
    "HF_DATASETS_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
    "WANDB_MODE": "offline",
    "WANDB_DISABLED": "true",
    "AWS_EC2_METADATA_DISABLED": "true",
    "HTTP_PROXY": "http://127.0.0.1:9",
    "HTTPS_PROXY": "http://127.0.0.1:9",
    "ALL_PROXY": "http://127.0.0.1:9",
    "NO_PROXY": "",
})

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
from IPython.display import display
from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection

SEED = 42
rng = np.random.default_rng(SEED)

DATA_PATH = Path(
    "notebooks/scPRINT-2-repro-notebooks/data/task_3_embed.h5ad"
)
RESULT_ROOT = Path("data/results/cross_species_embedding")
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

BATCH_KEY = "orig.ident"
LABEL_KEY = "cell_type_ontology_term_id"
PIG_ONTOLOGY_ID = "NCBITaxon:9823"
ORTHOLOG_PATH = Path(
    "notebooks/scPRINT-2-repro-notebooks/data/"
    "mouse_to_pig_one2one_orthologs.tsv"
)

for required_path in (DATA_PATH, ORTHOLOG_PATH):
    if not required_path.exists():
        raise FileNotFoundError(required_path)

import torch
import scprint2
import scdataloader.collator as scd_collator
from scprint2 import scPRINT2
from scprint2.model import utils as scprint_model_utils
from scprint2.tasks import Embedder

CHECKPOINT_PATH = Path("/lustre/fswork/projects/rech/xeg/uat95fg/ujzjjsi3.ckpt")
OUTPUT_PATH = RESULT_ROOT / "scprint_ujzjjsi3_pig_embeddings.h5ad"
SCORE_PATH = RESULT_ROOT / "scprint_ujzjjsi3_pig_scib.csv"

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(CHECKPOINT_PATH)
if not torch.cuda.is_available():
    raise RuntimeError("This notebook requires a Slurm GPU allocation")

print(f"SCPRINT2_SOURCE={scprint2.__file__}")
torch.set_float32_matmul_precision("medium")

→ connected lamindb: jkobject/scprint2


/lustre/fswork/projects/rech/xeg/uat95fg/simpler_flash/src/simpler_flash/layer_norm.py:1044: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/lustre/fswork/projects/rech/xeg/uat95fg/simpler_flash/src/simpler_flash/layer_norm.py:1107: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd


SCPRINT2_SOURCE=/lustre/fswork/projects/rech/xeg/uat95fg/scPRINT2-cross-species-rerun-8b06f63/scprint2/__init__.py


## Remap the mouse-gene input to pig checkpoint genes

Only Ensembl BioMart orthologs marked `ortholog_one2one` with confidence 1 are
used. The resulting matrix is ordered exactly like the pig vocabulary stored
inside the checkpoint; no ontology or gene metadata is fetched at runtime.


In [2]:
source_da = sc.read_h5ad(DATA_PATH)
model = scPRINT2.load_from_checkpoint(
    CHECKPOINT_PATH,
    precpt_gene_emb=None,
    gene_pos_file=None,
    map_location="cpu",
)

mapping = pd.read_csv(ORTHOLOG_PATH, sep="\t", dtype=str).rename(
    columns={
        "Gene stable ID": "mouse_ensembl_id",
        "Pig gene stable ID": "pig_ensembl_id",
    }
)
required_columns = {
    "mouse_ensembl_id",
    "pig_ensembl_id",
    "Pig homology type",
    "Pig orthology confidence [0 low, 1 high]",
}
if not required_columns.issubset(mapping.columns):
    raise KeyError(f"Ortholog table lacks: {required_columns - set(mapping.columns)}")
mapping = mapping.loc[
    (mapping["Pig homology type"] == "ortholog_one2one")
    & (mapping["Pig orthology confidence [0 low, 1 high]"] == "1")
].dropna(subset=["mouse_ensembl_id", "pig_ensembl_id"])
if mapping["mouse_ensembl_id"].duplicated().any() or mapping["pig_ensembl_id"].duplicated().any():
    raise ValueError("Expected a one-to-one mouse-to-pig ortholog mapping")

mouse_by_pig = mapping.set_index("pig_ensembl_id")["mouse_ensembl_id"].to_dict()
pig_checkpoint_genes = list(model._genes[PIG_ONTOLOGY_ID])
pig_genes = [
    gene
    for gene in pig_checkpoint_genes
    if gene in mouse_by_pig and mouse_by_pig[gene] in source_da.var_names
]
mouse_genes = [mouse_by_pig[gene] for gene in pig_genes]
if not pig_genes:
    raise RuntimeError("No mouse-to-pig ortholog overlaps the input and checkpoint")

mapped_counts = source_da[:, mouse_genes].X
if not sp.issparse(mapped_counts):
    mapped_counts = sp.csr_matrix(mapped_counts)
mapped_counts = mapped_counts.tocoo()
pig_positions = {gene: index for index, gene in enumerate(pig_checkpoint_genes)}
mapped_columns = np.fromiter(
    (pig_positions[pig_genes[index]] for index in mapped_counts.col),
    dtype=np.int64,
    count=mapped_counts.nnz,
)
pig_counts = sp.csr_matrix(
    (mapped_counts.data, (mapped_counts.row, mapped_columns)),
    shape=(source_da.n_obs, len(pig_checkpoint_genes)),
)
da = ad.AnnData(
    X=pig_counts,
    obs=source_da.obs.copy(),
    var=pd.DataFrame(
        index=pd.Index(pig_checkpoint_genes, name="ensembl_gene_id")
    ),
)
da.var["ensembl_gene_id"] = da.var_names.astype(str)
da.obs["organism_ontology_term_id"] = PIG_ONTOLOGY_ID
if set(da.obs["organism_ontology_term_id"].astype(str)) != {PIG_ONTOLOGY_ID}:
    raise ValueError("All cells must use the pig organism token")
print(f"Pig ortholog genes used: {len(pig_genes)} / {len(pig_checkpoint_genes)}")


def checkpoint_gene_table(organisms):
    """Build Collator metadata only from genes stored in the checkpoint."""
    if not isinstance(model._genes, dict):
        raise TypeError("Expected a checkpoint-embedded gene dictionary")
    if isinstance(organisms, str):
        organisms = [organisms]
    frames = [
        pd.DataFrame(
            {"organism": organism},
            index=pd.Index(model._genes[organism], name="ensembl_gene_id"),
        )
        for organism in organisms
    ]
    return pd.concat(frames)


def skip_ontology_translation(values, class_name):
    """Keep checkpoint ontology IDs without querying Bionty."""
    return None


scd_collator.load_genes = checkpoint_gene_table
scprint_model_utils.translate = skip_ontology_translation
model = model.to("cuda").eval()
print("Model organisms:", model.organisms)


FYI: scPRINT2 is not attached to a `Trainer`.


Pig ortholog genes used: 13714 / 22002


Model organisms: ['NCBITaxon:10090', 'NCBITaxon:10181', 'NCBITaxon:3702', 'NCBITaxon:4577', 'NCBITaxon:6239', 'NCBITaxon:7227', 'NCBITaxon:7955', 'NCBITaxon:9031', 'NCBITaxon:9483', 'NCBITaxon:9544', 'NCBITaxon:9598', 'NCBITaxon:9606', 'NCBITaxon:9823', 'NCBITaxon:9913', 'NCBITaxon:9940', 'NCBITaxon:9986']


## Generate scPRINT cell embeddings

In [3]:
if OUTPUT_PATH.exists():
    output_da = sc.read_h5ad(OUTPUT_PATH)
else:
    embedder = Embedder(
        how="random expr",
        max_len=3200,
        num_workers=min(4, int(os.environ.get("SLURM_CPUS_PER_TASK", "4"))),
        doclass=False,
        pred_embedding=["all"],
        doplot=False,
    )
    output_da, embedding_metrics = embedder(model, da.copy())
if "scprint_emb" not in output_da.obsm:
    raise KeyError("Embedder output has no obsm['scprint_emb']")
output_da.obsm["model_emb"] = np.asarray(output_da.obsm["scprint_emb"], dtype=np.float32)
output_da.write_h5ad(OUTPUT_PATH, compression="lzf")
output_da

predict epoch start


  0%|          | 0/425 [00:00<?, ?it/s]

  0%|          | 1/425 [00:10<1:11:47, 10.16s/it]

  0%|          | 2/425 [00:10<30:01,  4.26s/it]  

  1%|          | 3/425 [00:10<16:40,  2.37s/it]

...

logging the anndata


AnnData object with n_obs × n_vars = 27200 × 0
    obs: 'pred_cell_type_ontology_term_id', 'pred_tissue_ontology_term_id', 'pred_disease_ontology_term_id', 'pred_assay_ontology_term_id', 'pred_self_reported_ethnicity_ontology_term_id', 'pred_sex_ontology_term_id', 'pred_organism_ontology_term_id'
    obsm: 'scprint_emb_other', 'scprint_emb_cell_type_ontology_term_id', 'scprint_emb_tissue_ontology_term_id', 'scprint_emb_disease_ontology_term_id', 'scprint_emb_assay_ontology_term_id', 'scprint_emb_self_reported_ethnicity_ontology_term_id', 'scprint_emb_sex_ontology_term_id', 'scprint_emb_organism_ontology_term_id'


... storing 'organism_ontology_term_id' as categorical


too few cells to embed into a umap
too few cells to compute a clustering


AnnData object with n_obs × n_vars = 27200 × 22002
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'seurat_clusters', 'cell_type', 'batch', 'barcode', 'celltype', 'percent.mt', 'integrated_snn_res.1', 'NewCelltype', 'n_genes', 'organism_ontology_term_id', 'nnz', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'outlier', 'mt_outlier', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'pred_cell_type_ontology_term_id', 'pred_tissue_ontology_term_id', 'pred_disease_ontology_term_id', 'pred_assay_ontology_term_id', 'pred_self_reported_ethnicity_ontology_term_id', 'pred_sex_ontology_term_id', 'pred_organism_ontology_term_id'
    var: 'ensembl_gene_id'
    obsm: 'scprint_emb', 'scprint_emb_assay_ontology_term_id', 'scprint_emb_c

## scIB embedding scores

In [4]:
def build_benchmark_adata(
    expression_source: ad.AnnData,
    model_output: ad.AnnData,
    model_embedding_key: str,
) -> ad.AnnData:
    """Attach a model embedding to the full expression matrix and add common baselines."""
    if not expression_source.obs_names.equals(model_output.obs_names):
        raise RuntimeError("Expression source cell names/order differ from model output")
    if model_embedding_key not in model_output.obsm:
        raise KeyError(f"Missing model embedding: {model_embedding_key}")
    benchmark_adata = expression_source.copy()
    benchmark_adata.obsm[model_embedding_key] = np.asarray(
        model_output.obsm[model_embedding_key], dtype=np.float32
    )
    sc.pp.normalize_total(benchmark_adata, target_sum=1e4)
    sc.pp.log1p(benchmark_adata)
    sc.tl.pca(
        benchmark_adata,
        n_comps=50,
        svd_solver="arpack",
        use_highly_variable=False,
    )
    benchmark_adata.obsm["random"] = rng.random(
        benchmark_adata.obsm["X_pca"].shape, dtype=np.float32
    )
    return benchmark_adata


def benchmark_embeddings(adata: ad.AnnData, model_embedding_key: str):
    """Run the original cross-species scIB comparison and return unscaled scores."""
    for key in (BATCH_KEY, LABEL_KEY):
        if key not in adata.obs:
            raise KeyError(f"Missing obs column: {key}")
    benchmark = Benchmarker(
        adata,
        batch_key=BATCH_KEY,
        label_key=LABEL_KEY,
        embedding_obsm_keys=[model_embedding_key, "X_pca", "random"],
        pre_integrated_embedding_obsm_key="X_pca",
        bio_conservation_metrics=BioConservation(),
        batch_correction_metrics=BatchCorrection(),
        n_jobs=min(10, int(os.environ.get("SLURM_CPUS_PER_TASK", "10"))),
    )
    benchmark.benchmark()
    return benchmark.get_results(min_max_scale=False)

In [5]:
benchmark_da = build_benchmark_adata(
    sc.read_h5ad(DATA_PATH),
    output_da,
    "model_emb",
)
results = benchmark_embeddings(benchmark_da, "model_emb")
results.to_csv(SCORE_PATH)
output_da.obsm["X_pca"] = benchmark_da.obsm["X_pca"]
output_da.obsm["random"] = benchmark_da.obsm["random"]
output_da.write_h5ad(OUTPUT_PATH, compression="lzf")
display(results)
print("Scores:", SCORE_PATH)

/lustre/fswork/projects/rech/xeg/uat95fg/scPRINT/.venv/lib/python3.12/site-packages/scanpy/preprocessing/_pca/__init__.py:227: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  mask_var_param, mask_var = _handle_mask_var(


Computing neighbors:   0%|          | 0/3 [00:00<?, ?it/s]

Computing neighbors:  33%|███▎      | 1/3 [00:21<00:42, 21.09s/it]

Computing neighbors:  67%|██████▋   | 2/3 [00:28<00:12, 12.87s/it]

Computing neighbors: 100%|██████████| 3/3 [00:33<00:00,  9.30s/it]

...

Metrics:  80%|████████  | 8/10 [00:39<00:07,  3.69s/it, Batch correction: pcr_comparison]    

Embeddings: 100%|██████████| 3/3 [04:37<00:00, 74.98s/it] 

Embeddings: 100%|██████████| 3/3 [04:37<00:00, 92.40s/it]

,Isolated labels,KMeans NMI,KMeans ARI,Silhouette label,cLISI,BRAS,iLISI,KBET,Graph connectivity,PCR comparison,Batch correction,Bio conservation,Total
Embedding,,,,,,,,,,,,,
model_emb,0.434108,0.138568,0.072366,0.43501,0.857625,0.699575,0.247434,0.127649,0.546482,0.940349,0.512297,0.387535,0.43744
X_pca,0.600084,0.680804,0.578622,0.598215,1.0,0.496108,0.0,0.00017,0.898009,0.0,0.278858,0.691545,0.52647
random,0.497309,0.000953,-0.000088,0.497098,0.605362,0.993115,0.858831,0.939754,0.091398,0.999879,0.776595,0.320127,0.502714
Metric Type,Bio conservation,Bio conservation,Bio conservation,Bio conservation,Bio conservation,Batch correction,Batch correction,Batch correction,Batch correction,Batch correction,Aggregate score,Aggregate score,Aggregate score


Scores: data/results/cross_species_embedding/scprint_ujzjjsi3_pig_scib.csv
